# Feature Engineering — Analízis Notebook

**Asset:** `solusdt` | **Targetok:** `long_mfe_fw60`, `short_mfe_fw60`

## Metodológiai háttér

- [`_doc_/3200_features.md`](../../_doc_/3200_features.md) — Feature layer: lookahead-mentesség, t-1 lag, warmup, csoportok (25 db)
- [`_doc_/3000_modelling.md`](../../_doc_/3000_modelling.md) — Modeling pipeline áttekintés

A notebook a `src/modeling/feature_engineering/` library 4 analízis lépését futtatja
sorban: **quality → target_relation → redundancy → stability**. Az eredmény
determinisztikusan kerül a `feature_set.json`-be a konfigurált küszöbök alapján —
nincs manuális szerkesztés szükséges.

In [ ]:
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import duckdb
import polars as pl

# project root keresés pyproject.toml alapján
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))

import utils
from modeling.feature_engineering import (
    FeatureEngineeringConfig,
    analyze_quality,
    analyze_target_relation,
    analyze_redundancy,
    analyze_stability,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
ASSET_ID = "solusdt"
RUN_ID   = datetime.now().strftime("run_%Y%m%d_%H%M%S")

asset_cfg  = utils.load_asset_config(ASSET_ID)
db_path    = asset_cfg["database"]["db_path"]
data_dir   = Path(asset_cfg["database"]["data_dir"])
output_dir = data_dir / "feature_engineering" / RUN_ID
cfg        = FeatureEngineeringConfig(asset_id=ASSET_ID, run_id=RUN_ID)

conn   = duckdb.connect(db_path, read_only=True)
n_rows = conn.execute("SELECT COUNT(*) FROM quant_train").fetchone()[0]

print(f"asset_id  : {ASSET_ID}")
print(f"run_id    : {RUN_ID}")
print(f"db        : {db_path}")
print(f"output    : {output_dir}")
print(f"rows      : {n_rows:,}")

---

## 1. Quality — Univariáns minőség szűrés

Mit vizsgálunk: minden `feat_*` oszlopra kiszámítjuk a null arányt, az inf értékek
arányát, a varianciát és az outlier hányadot (|z-score| > 3). A döntés:

| Döntés | Feltétel |
|--------|----------|
| `drop` | `null_rate > 0.01` VAGY `inf_rate > 0.001` VAGY `variance < 1e-8` |
| `review` | `outlier_ratio > 0.05` (és nincs drop ok) |
| `keep` | minden más |

Kontextus: a feature layer t-1 lag-ot alkalmaz, ezért az első 1441 sor null
(`prev_session` warmup). Ezek a nullák **nem okoznak minőségi problémát** — a
sampling `lookback_end_ts` offset garantálja, hogy warmup sorok nem kerülnek
a tanítási ablakba.

In [ ]:
quality_df = analyze_quality(conn, cfg)
quality_df

In [ ]:
vc = quality_df["decision"].value_counts().sort("decision")
labels = vc["decision"].to_list()
counts = vc["count"].to_list()
colors = {"drop": "#f44336", "keep": "#4caf50", "review": "#ff9800"}

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(labels, counts, color=[colors.get(l, "#90a4ae") for l in labels])
ax.set_title("Quality döntések")
ax.set_ylabel("Feature count")
for i, v in enumerate(counts):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
not_keep_q = quality_df.filter(pl.col("decision") != "keep").select(
    ["feature", "null_rate", "inf_rate", "variance", "outlier_ratio", "decision", "drop_reason"]
).sort("decision")
print(f"Nem-keep features: {len(not_keep_q)}")
not_keep_q

---

## 2. Target Relation — Szignálerősség

Mit vizsgálunk: minden `feat_*` × target párra Pearson (`CORR`) és Spearman
(RANK-alapú) korrelációt számítunk a `quant_train` táblán. `signal_proxy = |ρ_spearman|`.

| Döntés | Feltétel |
|--------|----------|
| `leakage` | `|ρ| > 0.95` — gyanús, jövőbeli adat szivárgás |
| `weak` | `|ρ| < 0.01` — nincs érdemi szignál |
| `keep` | minden más |

Egy feature akkor kerül ki, ha **mindkét** targetre `weak` (vagy `leakage`).
Ha csak az egyik targettel gyenge, megtartjuk.

Megjegyzés: a `quant_train` csak NULL-mentes target sorokat tartalmaz (ezt a
build pipeline garantálja), ezért nincs szükség külön szűrésre.

In [ ]:
relation_df = analyze_target_relation(conn, cfg)
relation_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, target in zip(axes, cfg.target_cols):
    sub = (
        relation_df
        .filter(pl.col("target") == target)
        .sort("signal_proxy", descending=True)
    )
    top20 = sub.head(20)
    ax.barh(top20["feature"].to_list(), top20["signal_proxy"].to_list(), color="#2196f3")
    ax.set_title(f"Top 20 signal_proxy — {target}")
    ax.set_xlabel("|ρ_spearman|")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
not_keep_r = relation_df.filter(pl.col("decision") != "keep").sort(["feature", "target"])
print(f"Nem-keep (feature, target) párok: {len(not_keep_r)}")
not_keep_r

---

## 3. Redundancy — Korrelációs klaszterezés

Mit vizsgálunk: a `feat_*` oszlopokat Pearson korrelációs mátrix alapján
klaszterezzük (union-find algoritmus). Ha két feature |Pearson r| ≥ `pearson_cluster_thr`
(0.95), egy klaszterbe kerülnek. Klaszterenként egy reprezentatív marad (`keep`),
a többi `drop` lesz.

A korrelációs mátrix **500 000 véletlenszerű sorból** számolódik (sampling).
Ez elegendő a 0.95 küszöbű korrelációk megbízható detektálásához ~3M soron,
miközben a RAM-igény ~830 MB marad.

Reprezentatív = **legkisebb indexű** feature a klaszterben (megérkezési sorrend
alapján). A downstream `02_hyper_param_search.py` LightGBM feature importance-on
keresztül tovább szűrhet.

In [ ]:
redundancy_df = analyze_redundancy(conn, cfg)
redundancy_df.head(10)

In [ ]:
n_clusters = redundancy_df["cluster_id"].n_unique()
n_multi    = (
    redundancy_df.group_by("cluster_id")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .height
)
n_dropped  = (redundancy_df["decision"] == "drop").sum()

print(f"Klaszterek összesen     : {n_clusters}")
print(f"Klaszterek >1 taggal    : {n_multi}")
print(f"Redundáns (drop) feature: {n_dropped}")

redundancy_df.filter(pl.col("decision") == "drop").select(
    ["feature", "cluster_id", "max_pearson", "drop_reason"]
).sort("cluster_id")

---

## 4. Stability — Időbeli stabilitás

Mit vizsgálunk: az adatot 90 napos, nem-átfedő időablakokra osztjuk. Minden
(feature, bucket) párra Spearman korrelációt számítunk mindkét targettel, és
összehasonlítjuk a globális baseline-nal.

`drift = |ρ_bucket − ρ_baseline|`

| Flag | Feltétel | Hatás |
|------|----------|-------|
| `stable` | drift ≤ 0.15 | megtartjuk |
| `review` | 0.15 < drift ≤ 0.30 | megtartjuk, jelölt |
| `unstable` | drift > 0.30, nem az utolsó 2 bucket | megtartjuk |
| `decayed` | drift > 0.30 **és** az utolsó 2 bucket | **kizárjuk** |

A `decayed` flag azt jelenti, hogy a feature a legutóbbi adatszakaszban elveszítette
prediktív kapcsolatát a targettel — a modell live-on már nem bízhat benne.

In [ ]:
stability_df = analyze_stability(conn, cfg)
print(f"(feature, bucket) sorok: {len(stability_df):,}")
stability_df.head(10)

In [ ]:
problematic = (
    stability_df
    .filter(pl.col("stability_flag").is_in(["decayed", "unstable"]))
    ["feature"].unique().to_list()
)

if problematic:
    sample_feats = sorted(problematic)[:6]
    fig, axes = plt.subplots(len(sample_feats), 1, figsize=(10, 3 * len(sample_feats)))
    if len(sample_feats) == 1:
        axes = [axes]
    for ax, feat in zip(axes, sample_feats):
        sub = stability_df.filter(pl.col("feature") == feat).sort("bucket_idx")
        ax.plot(sub["bucket_idx"].to_list(), sub["drift_long"].to_list(),
                marker="o", label="drift_long")
        ax.plot(sub["bucket_idx"].to_list(), sub["drift_short"].to_list(),
                marker="s", label="drift_short")
        ax.axhline(cfg.max_drift_threshold, color="red", linestyle="--",
                   linewidth=1, label=f"threshold ({cfg.max_drift_threshold})")
        ax.set_title(feat, fontsize=9)
        ax.legend(fontsize=7)
        ax.set_xlabel("bucket_idx")
        ax.set_ylabel("drift")
    plt.tight_layout()
    plt.show()
else:
    print("Nincs decayed/unstable feature.")

In [ ]:
decayed_features = (
    stability_df
    .filter(pl.col("stability_flag") == "decayed")
    ["feature"].unique().to_list()
)
print(f"Decayed (kizárt) features: {len(decayed_features)}")

if decayed_features:
    (
        stability_df
        .filter(pl.col("feature").is_in(decayed_features))
        .group_by("feature")
        .agg([
            pl.col("drift_long").max().alias("max_drift_long"),
            pl.col("drift_short").max().alias("max_drift_short"),
            pl.col("stability_flag")
              .filter(pl.col("stability_flag") == "decayed")
              .len().alias("decayed_buckets"),
        ])
        .sort("max_drift_long", descending=True)
    )

---

## Output — feature_set.json

A 4 lépés eredményét kombináljuk. Egy feature a **selected** listába kerül,
ha az összes feltétel teljesül:

1. **quality** → `keep`
2. **target_relation** → legalább egy targetre `keep`
3. **redundancy** → `keep` (reprezentatív a klaszterben)
4. **stability** → nincs `decayed` bucket

Minden más `dropped` (okkal) vagy `review` (marginális, de nem kizárt) lesz.
A JSON a `selected` listát adja tovább a `00_create_sample.py`-nak.

In [ ]:
conn.close()
output_dir.mkdir(parents=True, exist_ok=True)

all_features = quality_df["feature"].to_list()
created_at   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# --- quality ---
quality_drop   = {r["feature"]: r["drop_reason"] for r in quality_df.iter_rows(named=True) if r["decision"] == "drop"}
quality_review = {r["feature"] for r in quality_df.iter_rows(named=True) if r["decision"] == "review"}

# --- relation: keep ha legalább egy targetre 'keep' ---
relation_keep = {r["feature"] for r in relation_df.iter_rows(named=True) if r["decision"] == "keep"}
relation_leak = {r["feature"] for r in relation_df.iter_rows(named=True) if r["decision"] == "leakage"}

# --- redundancy ---
redundancy_drop = {r["feature"]: r["drop_reason"] for r in redundancy_df.iter_rows(named=True) if r["decision"] == "drop"}

# --- stability ---
decayed_set  = {r["feature"] for r in stability_df.iter_rows(named=True) if r["stability_flag"] == "decayed"}
unstable_set = {r["feature"] for r in stability_df.iter_rows(named=True) if r["stability_flag"] in ("unstable", "review")} - decayed_set

selected: list[str]  = []
dropped:  list[dict] = []
review:   list[str]  = []

for feat in all_features:
    reasons: list[str] = []
    if feat in quality_drop:
        reasons.append(f"quality: {quality_drop[feat]}")
    if feat in relation_leak:
        reasons.append("relation: leakage suspect")
    if feat not in relation_keep and feat not in relation_leak:
        reasons.append("relation: weak signal across all targets")
    if feat in redundancy_drop:
        reasons.append(f"redundancy: {redundancy_drop[feat]}")
    if feat in decayed_set:
        reasons.append("stability: decayed in recent buckets")

    is_review = (feat in quality_review or feat in unstable_set) and not reasons

    if reasons:
        dropped.append({"col": feat, "reason": " | ".join(reasons)})
    elif is_review:
        review.append(feat)
    else:
        selected.append(feat)

fs = {
    "run_id"     : cfg.run_id,
    "asset_id"   : cfg.asset_id,
    "created_at" : created_at,
    "target_cols": list(cfg.target_cols),
    "selected"   : selected,
    "dropped"    : dropped,
    "review"     : review,
    "thresholds" : {
        "max_null_rate"        : cfg.max_null_rate,
        "max_inf_rate"         : cfg.max_inf_rate,
        "min_variance"         : cfg.min_variance,
        "max_outlier_ratio"    : cfg.max_outlier_ratio,
        "min_spearman_abs"     : cfg.min_spearman_abs,
        "max_spearman_leakage" : cfg.max_spearman_leakage,
        "pearson_cluster_thr"  : cfg.pearson_cluster_thr,
        "redundancy_max_rows"  : cfg.redundancy_max_rows,
        "stability_bucket_days": cfg.stability_bucket_days,
        "max_drift_threshold"  : cfg.max_drift_threshold,
    },
}

json_path = output_dir / "feature_set.json"
json_path.write_text(json.dumps(fs, indent=2), encoding="utf-8")

print(f"Selected : {len(selected)}")
print(f"Dropped  : {len(dropped)}")
print(f"Review   : {len(review)}")
print(f"")
print(f"Output   : {output_dir}")
print(f"           feature_set.json")

In [ ]:
# Summary vizualizáció
fig, ax = plt.subplots(figsize=(5, 3))
categories = ["selected", "dropped", "review"]
values     = [len(fs["selected"]), len(fs["dropped"]), len(fs["review"])]
bar_colors = ["#4caf50", "#f44336", "#ff9800"]
ax.bar(categories, values, color=bar_colors)
ax.set_title("Feature szelekció végeredmény")
ax.set_ylabel("Feature count")
for i, v in enumerate(values):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()